# Observation Period — Allscripts Sunrise (SCM)

**OMOP CDM v5.4 — `observation_period` table**

### Source
- `_exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientvisit`

### Target
- `_exponent.omop.observation_period`

### Strategy
- Derive observation periods from visit records in `dbo_cv3clientvisit`
- Use `AdmitDtm` as the start date anchor
- Use `COALESCE(DischargeDtm, CloseDtm, TouchedWhen)` for end date
- Option A: One observation period per patient (MIN/MAX aggregation)
- Option B: Multiple observation periods per patient (gap-based splitting)
- `period_type_concept_id = 32817` (EHR encounter record)

### Dependencies
- `source_to_person` mapping must be populated for Allscripts SCM patients


### Date
- 2026-02-10

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Load Source Data

In [0]:
source_table = "_exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientvisit"

df_source = spark.table(source_table)

total_rows = df_source.count()
distinct_patients = df_source.select("ClientGUID").distinct().count()

print(f"Total rows: {total_rows:,}")
print(f"Distinct patients (ClientGUID): {distinct_patients:,}")

## EDA: Date Coverage
Check null counts, date ranges, and completeness for the key date columns.

In [0]:
# date_cols = ["AdmitDtm", "DischargeDtm", "CloseDtm", "PlannedDischargeDtm"]

# date_stats = df_source.select(
#     F.lit(total_rows).alias("total_rows"),
#     *[
#         expr
#         for col_name in date_cols
#         for expr in [
#             F.count(F.col(col_name)).alias(f"{col_name}_non_null"),
#             F.sum(F.when(F.col(col_name).isNull(), 1).otherwise(0)).alias(f"{col_name}_null"),
#             F.round(
#                 F.count(F.col(col_name)) / F.lit(total_rows) * 100, 2
#             ).alias(f"{col_name}_pct"),
#             F.min(F.col(col_name)).alias(f"{col_name}_min"),
#             F.max(F.col(col_name)).alias(f"{col_name}_max"),
#         ]
#     ]
# )

# display(date_stats)

## EDA: Filter Candidates
Show value distributions for `Active`, `VisitStatus`, and `InternalVisitStatus` to determine WHERE clause filters.

In [0]:
# print("=== Active flag distribution ===")
# display(
#     df_source.groupBy("Active")
#     .agg(
#         F.count("*").alias("count"),
#         F.round(F.count("*") / F.lit(total_rows) * 100, 2).alias("pct")
#     )
#     .orderBy(F.desc("count"))
# )

# print("\n=== VisitStatus distribution ===")
# display(
#     df_source.groupBy("VisitStatus")
#     .agg(
#         F.count("*").alias("count"),
#         F.round(F.count("*") / F.lit(total_rows) * 100, 2).alias("pct")
#     )
#     .orderBy(F.desc("count"))
# )

# print("\n=== InternalVisitStatus distribution ===")
# display(
#     df_source.groupBy("InternalVisitStatus")
#     .agg(
#         F.count("*").alias("count"),
#         F.round(F.count("*") / F.lit(total_rows) * 100, 2).alias("pct")
#     )
#     .orderBy(F.desc("count"))
# )

## EDA: Visit Types
Distributions for `TypeCode` and `CareLevelCode`.

In [0]:
# print("=== TypeCode distribution ===")
# display(
#     df_source.groupBy("TypeCode")
#     .agg(
#         F.count("*").alias("count"),
#         F.round(F.count("*") / F.lit(total_rows) * 100, 2).alias("pct")
#     )
#     .orderBy(F.desc("count"))
# )

# print("\n=== CareLevelCode distribution ===")
# display(
#     df_source.groupBy("CareLevelCode")
#     .agg(
#         F.count("*").alias("count"),
#         F.round(F.count("*") / F.lit(total_rows) * 100, 2).alias("pct")
#     )
#     .orderBy(F.desc("count"))
# )

## EDA: End Date Fallback Analysis
Validate the COALESCE fallback strategy: `COALESCE(DischargeDtm, CloseDtm, TouchedWhen)`.

Quantify how many rows fall into each tier of the fallback chain.

In [0]:
# fallback_analysis = df_source.select(
#     F.count("*").alias("total_rows"),
#     F.sum(
#         F.when(F.col("DischargeDtm").isNotNull(), 1).otherwise(0)
#     ).alias("has_discharge_dtm"),
#     F.sum(
#         F.when(
#             F.col("DischargeDtm").isNull() & F.col("CloseDtm").isNotNull(), 1
#         ).otherwise(0)
#     ).alias("fallback_to_close_dtm"),
#     F.sum(
#         F.when(
#             F.col("DischargeDtm").isNull()
#             & F.col("CloseDtm").isNull()
#             & F.col("TouchedWhen").isNotNull(),
#             1,
#         ).otherwise(0)
#     ).alias("fallback_to_touched_when"),
#     F.sum(
#         F.when(
#             F.col("DischargeDtm").isNull()
#             & F.col("CloseDtm").isNull()
#             & F.col("TouchedWhen").isNull(),
#             1,
#         ).otherwise(0)
#     ).alias("all_three_null"),
# )

# display(fallback_analysis)

## EDA: Gap Analysis
For each patient, calculate the number of days between consecutive visits (ordered by `AdmitDtm`).

This determines whether we need a single observation period per patient (Option A) or multiple gap-split periods (Option B).

In [0]:
# # Filter to valid visits and compute per-patient visit ordering
# w_patient = Window.partitionBy("ClientGUID").orderBy("AdmitDtm")

# df_gaps = (
#     df_source.filter(
#         (F.col("ClientGUID").isNotNull()) & (F.col("AdmitDtm").isNotNull())
#     )
#     .withColumn("prev_end", F.lag(
#         F.coalesce(
#             F.col("DischargeDtm"), F.col("CloseDtm"), F.col("TouchedWhen")
#         )
#     ).over(w_patient))
#     .filter(F.col("prev_end").isNotNull())
#     .withColumn(
#         "gap_days",
#         F.datediff(F.col("AdmitDtm"), F.col("prev_end"))
#     )
# )

# # Summary statistics
# gap_stats = df_gaps.select(
#     F.count("*").alias("total_gaps"),
#     F.round(F.avg("gap_days"), 1).alias("avg_gap_days"),
#     F.expr("percentile_approx(gap_days, 0.5)").alias("median_gap_days"),
#     F.expr("percentile_approx(gap_days, 0.95)").alias("p95_gap_days"),
#     F.max("gap_days").alias("max_gap_days"),
#     F.sum(F.when(F.col("gap_days") > 365, 1).otherwise(0)).alias("gaps_over_365d"),
#     F.round(
#         F.sum(F.when(F.col("gap_days") > 365, 1).otherwise(0))
#         / F.count("*")
#         * 100,
#         2,
#     ).alias("pct_gaps_over_365d"),
# )

# display(gap_stats)

---
# Transformation

## Option A: Simple — One Observation Period per Patient
Aggregates `MIN(AdmitDtm)` and `MAX(COALESCE(DischargeDtm, CloseDtm, TouchedWhen))` per patient into a single observation period.

In [0]:
# source = "allscripts_scm"

# silver_observation_period_a = spark.sql(f"""
# WITH filtered_visits AS (
#     SELECT
#         ClientGUID,
#         AdmitDtm,
#         COALESCE(DischargeDtm, CloseDtm, TouchedWhen) AS end_dtm
#     FROM {source_table}
#     WHERE ClientGUID IS NOT NULL
#       AND AdmitDtm IS NOT NULL
#       AND Active = TRUE
# ),

# patient_observation_window AS (
#     SELECT
#         ClientGUID,
#         MIN(DATE(AdmitDtm)) AS observation_period_start_date,
#         MAX(DATE(COALESCE(end_dtm, AdmitDtm))) AS observation_period_end_date
#     FROM filtered_visits
#     WHERE AdmitDtm >= '1900-01-01'
#       AND AdmitDtm <= CURRENT_DATE()
#     GROUP BY ClientGUID
# )

# SELECT
#     pow.observation_period_start_date,
#     pow.observation_period_end_date,
#     32817 AS period_type_concept_id,
#     CONCAT('{source}', ' | ', CAST(pow.ClientGUID AS STRING)) AS person_source_value,
#     CONCAT('{source}', ' | ', CAST(pow.ClientGUID AS STRING)) AS observation_period_source_value,
#     '{source}' AS source_system
# FROM patient_observation_window pow
# INNER JOIN _exponent.omop_mapping.source_to_person stp
#     ON stp.person_source_value = CONCAT('{source}', ' | ', CAST(pow.ClientGUID AS STRING))
#    AND stp.active_flag = TRUE
# WHERE pow.observation_period_start_date <= pow.observation_period_end_date
# """)

# print(f"Option A — observation periods: {silver_observation_period_a.count():,}")
# display(silver_observation_period_a)

## Option B: Multiple Observation Periods per Patient (Gap-Based Splitting)
Uses a configurable gap threshold (default 365 days) to split patients with long gaps into separate observation periods.

Window functions detect gaps between consecutive visits; a cumulative sum of gap flags assigns period groups.

In [0]:
source = "allscripts_scm"
GAP_THRESHOLD_DAYS = 365

silver_observation_period_b = spark.sql(f"""
WITH filtered_visits AS (
    SELECT
        ClientGUID,
        AdmitDtm,
        COALESCE(DischargeDtm, CloseDtm, TouchedWhen, AdmitDtm) AS end_dtm
    FROM {source_table}
    WHERE ClientGUID IS NOT NULL
      AND AdmitDtm IS NOT NULL
      AND VisitStatus != 'CAN'
      AND AdmitDtm >= '1900-01-01'
      AND AdmitDtm <= CURRENT_DATE()
),

visit_with_prev AS (
    SELECT
        ClientGUID,
        AdmitDtm,
        end_dtm,
        LAG(end_dtm) OVER (
            PARTITION BY ClientGUID ORDER BY AdmitDtm
        ) AS prev_end_dtm
    FROM filtered_visits
),

visit_with_gap_flag AS (
    SELECT
        *,
        CASE
            WHEN prev_end_dtm IS NULL THEN 1
            WHEN DATEDIFF(AdmitDtm, prev_end_dtm) > {GAP_THRESHOLD_DAYS} THEN 1
            ELSE 0
        END AS new_period_flag
    FROM visit_with_prev
),

visit_with_period_group AS (
    SELECT
        *,
        SUM(new_period_flag) OVER (
            PARTITION BY ClientGUID ORDER BY AdmitDtm
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS period_group
    FROM visit_with_gap_flag
),

grouped_periods AS (
    SELECT
        ClientGUID,
        period_group,
        MIN(DATE(AdmitDtm)) AS observation_period_start_date,
        MAX(DATE(end_dtm)) AS observation_period_end_date
    FROM visit_with_period_group
    GROUP BY ClientGUID, period_group
)

SELECT
    gp.observation_period_start_date,
    gp.observation_period_end_date,
    32817 AS period_type_concept_id,
    CONCAT('{source}', ' | ', CAST(gp.ClientGUID AS STRING)) AS person_source_value,
    CONCAT('{source}', ' | ', CAST(gp.ClientGUID AS STRING), ' | ', CAST(gp.period_group AS STRING)) AS observation_period_source_value,
    '{source}' AS source_system
FROM grouped_periods gp
INNER JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = CONCAT('{source}', ' | ', CAST(gp.ClientGUID AS STRING))
   AND stp.active_flag = TRUE
WHERE gp.observation_period_start_date <= gp.observation_period_end_date
""")

print(f"Option B — observation periods (gap threshold = {GAP_THRESHOLD_DAYS} days): {silver_observation_period_b.count():,}")
display(silver_observation_period_b)

In [0]:
# # Quick test — remove person join to verify data flows
# test_df = spark.sql(f"""
# WITH filtered_visits AS (
#     SELECT
#         ClientGUID,
#         AdmitDtm,
#         COALESCE(DischargeDtm, CloseDtm, TouchedWhen, AdmitDtm) AS end_dtm
#     FROM {source_table}
#     WHERE ClientGUID IS NOT NULL
#       AND AdmitDtm IS NOT NULL
#       AND VisitStatus != 'CAN'
#       AND AdmitDtm >= '1900-01-01'
#       AND AdmitDtm <= CURRENT_DATE()
# ),
# visit_with_prev AS (
#     SELECT ClientGUID, AdmitDtm, end_dtm,
#         LAG(end_dtm) OVER (PARTITION BY ClientGUID ORDER BY AdmitDtm) AS prev_end_dtm
#     FROM filtered_visits
# ),
# visit_with_gap_flag AS (
#     SELECT *,
#         CASE WHEN prev_end_dtm IS NULL THEN 1
#              WHEN DATEDIFF(AdmitDtm, prev_end_dtm) > {GAP_THRESHOLD_DAYS} THEN 1
#              ELSE 0 END AS new_period_flag
#     FROM visit_with_prev
# ),
# visit_with_period_group AS (
#     SELECT *, SUM(new_period_flag) OVER (
#         PARTITION BY ClientGUID ORDER BY AdmitDtm
#         ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
#     ) AS period_group
#     FROM visit_with_gap_flag
# ),
# grouped_periods AS (
#     SELECT ClientGUID, period_group,
#         MIN(DATE(AdmitDtm)) AS observation_period_start_date,
#         MAX(DATE(end_dtm)) AS observation_period_end_date
#     FROM visit_with_period_group
#     GROUP BY ClientGUID, period_group
# )
# SELECT COUNT(*) AS total_periods, COUNT(DISTINCT ClientGUID) AS distinct_patients
# FROM grouped_periods
# WHERE observation_period_start_date <= observation_period_end_date
# """)
# display(test_df)

## Select Transformation Option
Choose Option A or Option B based on the gap analysis results above, then assign to `silver_observation_period_df`.

In [0]:
# Toggle: set to 'A' or 'B'
OPTION = 'B'

if OPTION == 'A':
    silver_observation_period_df = silver_observation_period_a
elif OPTION == 'B':
    silver_observation_period_df = silver_observation_period_b
else:
    raise ValueError(f"Invalid OPTION: {OPTION}. Choose 'A' or 'B'.")

print(f"Using Option {OPTION}")
silver_observation_period_df.createOrReplaceTempView("silver_observation_period")

## Validation Checks

In [0]:
validation = silver_observation_period_df.select(
    F.count("*").alias("total_observation_periods"),
    F.countDistinct("person_source_value").alias("distinct_patients"),
    F.sum(
        F.when(F.col("observation_period_start_date").isNull(), 1).otherwise(0)
    ).alias("null_start_dates"),
    F.sum(
        F.when(F.col("observation_period_end_date").isNull(), 1).otherwise(0)
    ).alias("null_end_dates"),
    F.sum(
        F.when(
            F.col("observation_period_end_date") < F.col("observation_period_start_date"),
            1,
        ).otherwise(0)
    ).alias("end_before_start"),
    F.sum(
        F.when(F.col("person_source_value").isNull(), 1).otherwise(0)
    ).alias("null_person_source_values"),
    F.min("observation_period_start_date").alias("min_start_date"),
    F.max("observation_period_start_date").alias("max_start_date"),
    F.min("observation_period_end_date").alias("min_end_date"),
    F.max("observation_period_end_date").alias("max_end_date"),
)

display(validation)

---
## Write to Silver Layer

In [0]:
# -- Merge to Silver layer --
# Uncomment when ready to persist

# spark.sql("""
# MERGE INTO _exponent.omop_silver.observation_period AS t
# USING silver_observation_period AS s
# ON t.observation_period_source_value = s.observation_period_source_value
#
# WHEN MATCHED AND (
#      NOT (t.observation_period_start_date <=> s.observation_period_start_date)
#   OR NOT (t.observation_period_end_date <=> s.observation_period_end_date)
#   OR NOT (t.period_type_concept_id <=> s.period_type_concept_id)
#   OR NOT (t.person_source_value <=> s.person_source_value)
#   OR NOT (t.source_system <=> s.source_system)
# )
# THEN UPDATE SET
#   t.observation_period_start_date = s.observation_period_start_date,
#   t.observation_period_end_date   = s.observation_period_end_date,
#   t.period_type_concept_id        = s.period_type_concept_id,
#   t.person_source_value           = s.person_source_value,
#   t.source_system                 = s.source_system,
#   t.last_mod_tsp                  = current_timestamp()
#
# WHEN NOT MATCHED THEN
# INSERT (
#   observation_period_start_date,
#   observation_period_end_date,
#   period_type_concept_id,
#   person_source_value,
#   observation_period_source_value,
#   source_system,
#   last_mod_tsp
# )
# VALUES (
#   s.observation_period_start_date,
#   s.observation_period_end_date,
#   s.period_type_concept_id,
#   s.person_source_value,
#   s.observation_period_source_value,
#   s.source_system,
#   current_timestamp()
# );
# """)

## Insert Mapping Records

In [0]:
# -- Insert new mappings to source_to_observation_period --
# Uncomment when ready to persist

# spark.sql("""
# INSERT INTO _exponent.omop_mapping.source_to_observation_period (
#     source_system,
#     observation_period_source_value,
#     active_flag,
#     created_tsp,
#     last_mod_tsp
# )
# SELECT
#     s.source_system,
#     s.observation_period_source_value,
#     TRUE AS active_flag,
#     current_timestamp() AS created_tsp,
#     current_timestamp() AS last_mod_tsp
# FROM (
#     SELECT DISTINCT source_system, observation_period_source_value
#     FROM _exponent.omop_silver.observation_period
#     WHERE source_system = 'allscripts_scm'
# ) s
# LEFT ANTI JOIN _exponent.omop_mapping.source_to_observation_period x
#   ON s.observation_period_source_value = x.observation_period_source_value;
# """)

## Merge to Gold Layer

In [0]:
# -- Merge to Gold layer --
# Uncomment when ready to persist

# spark.sql("""
# MERGE INTO _exponent.omop.observation_period AS gold
# USING (
#   SELECT
#     sop.observation_period_id,
#     stp.person_id,
#     s.observation_period_start_date,
#     s.observation_period_end_date,
#     s.period_type_concept_id
#   FROM _exponent.omop_silver.observation_period s
#   JOIN _exponent.omop_mapping.source_to_observation_period sop
#     ON sop.observation_period_source_value = s.observation_period_source_value
#    AND sop.active_flag = TRUE
#   JOIN _exponent.omop_mapping.source_to_person stp
#     ON stp.person_source_value = s.person_source_value
#    AND stp.active_flag = TRUE
#   WHERE s.source_system = 'allscripts_scm'
# ) AS src
# ON gold.observation_period_id = src.observation_period_id
#
# WHEN MATCHED THEN UPDATE SET
#   gold.person_id                       = src.person_id,
#   gold.observation_period_start_date   = src.observation_period_start_date,
#   gold.observation_period_end_date     = src.observation_period_end_date,
#   gold.period_type_concept_id          = src.period_type_concept_id
#
# WHEN NOT MATCHED THEN INSERT (
#   observation_period_id,
#   person_id,
#   observation_period_start_date,
#   observation_period_end_date,
#   period_type_concept_id
# )
# VALUES (
#   src.observation_period_id,
#   src.person_id,
#   src.observation_period_start_date,
#   src.observation_period_end_date,
#   src.period_type_concept_id
# );
# """)